In [ ]:
import numpy as np
import lysis
import os

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.animation import FFMpegWriter, FuncAnimation
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

from IPython.display import HTML
plt.rcParams["animation.html"] = "jshtml"

In [ ]:
e = lysis.util.Experiment(os.path.join("..", "..", "data"), experiment_code="2023-01-31-1301")
e.read_file()
rng = np.random.default_rng()
f_file_code = ".f-array.dat"
p_file_code = ".p.npy"

In [ ]:
print(e)

In [ ]:
dt = np.dtype([('time', float), ('fiber', np.int32), ('r', float), ('rmicro', float), ('deg_time', float)])

In [ ]:
f_deg = np.fromfile(os.path.join(e.os_path, "deg" + f_file_code))
f_mol_location = np.fromfile(os.path.join(e.os_path, "m_loc" + f_file_code), dtype=np.int32)
f_mol_status = np.fromfile(os.path.join(e.os_path, "m_bound" + f_file_code), dtype=np.int32)
f_tsave = np.fromfile(os.path.join(e.os_path, "tsave" + f_file_code))
f_lysis_time = np.fromfile(os.path.join(e.os_path, "deg_tracker" + f_file_code), dtype=dt)

f_deg = f_deg.reshape(e.macro_params.total_trials, e.macro_params.number_of_saves, e.macro_params.total_edges)
f_mol_location = f_mol_location.reshape(e.macro_params.total_trials, e.macro_params.number_of_saves, e.macro_params.total_molecules) - 1
f_mol_status = f_mol_status.reshape(e.macro_params.total_trials, e.macro_params.number_of_saves, e.macro_params.total_molecules)
f_tsave = f_tsave.reshape(e.macro_params.total_trials, e.macro_params.number_of_saves)
#f_lysis_time = f_lysis_time.reshape(f_lysis_time.size // 5, 5)

In [ ]:
p_deg = np.load(os.path.join(e.os_path, "deg" + p_file_code))
p_mol_location = np.load(os.path.join(e.os_path, "m_loc" + p_file_code))
p_mol_status = np.load(os.path.join(e.os_path, "m_bound" + p_file_code))
p_tsave = np.load(os.path.join(e.os_path, "tsave" + p_file_code))
p_lysis_time = np.load(os.path.join(e.os_path, "deg_tracker" + p_file_code))

p_deg = np.expand_dims(p_deg, axis=0)
p_mol_location = np.expand_dims(p_mol_location, axis=0)
p_mol_status = np.expand_dims(p_mol_status, axis=0)
p_tsave = np.expand_dims(p_tsave, axis=0)
p_lysis_time = p_lysis_time.reshape(p_lysis_time.size // 5, 5)

true_edge = np.full(e.macro_params.rows*e.macro_params.full_row,
                    True,
                    np.bool_)
for k in range(e.macro_params.cols):
    true_edge[(e.macro_params.rows-1)*e.macro_params.full_row + 3*k] = False

In [ ]:
f_lysis_time.shape, p_lysis_time.shape

In [ ]:
f_mol_status = f_mol_status.astype(np.bool_)
f_mapped_deg = -f_deg
f_mapped_deg[f_deg == 0] = e.macro_params.total_time * 2
f_mapped_deg[f_deg == -1] = 0
p_mapped_deg = p_deg[:, :, true_edge]
for r in range(1):
    for s in range(e.macro_params.number_of_saves):
        p_mapped_deg[r, s][p_mapped_deg[r, s] > p_tsave[r, s]] = e.macro_params.total_time * 2
        p_deg[r, s][p_deg[r, s] > p_tsave[r, s]] = e.macro_params.total_time * 2

In [ ]:
def plot_coords(i, j):
    x = j
    y = i
    if j % 3 == 0:
        return x / 3.0, y + 0.5
    if j % 3 == 1:
        return (x - 1) / 3.0, y
    if j % 3 == 2:
        return (x - 2) / 3.0 + 0.5, y

In [ ]:
f_edge_index = np.empty(e.macro_params.total_edges, dtype=tuple)
for k in range(e.macro_params.total_edges):
    f_edge_index[k] = lysis.from_fortran_edge_index(k, e.macro_params.rows, e.macro_params.cols)
p_edge_index = np.empty(e.macro_params.rows*e.macro_params.full_row, dtype=tuple)
for k in range(e.macro_params.rows*e.macro_params.full_row):
    p_edge_index[k] = np.unravel_index(k, shape=(e.macro_params.rows, e.macro_params.full_row))

In [ ]:
f_edge_index[7790], p_edge_index[7794]

In [ ]:
f_square_deg = np.full((e.macro_params.number_of_saves, e.macro_params.rows, e.macro_params.full_row), 0.0, np.float_)
p_square_deg = np.full((e.macro_params.number_of_saves, e.macro_params.rows, e.macro_params.full_row), 0.0, np.float_)
for s in range(e.macro_params.number_of_saves):
    for k in range(e.macro_params.total_edges):
        f_square_deg[s, f_edge_index[k][0], f_edge_index[k][1]] = f_mapped_deg[0, s, k]
    for k in range(e.macro_params.rows*e.macro_params.full_row):
        p_square_deg[s, p_edge_index[k][0], p_edge_index[k][1]] = p_deg[0, s, k]

In [ ]:
f_square_deg.shape, p_square_deg.shape

In [ ]:
np.argwhere(np.abs(f_square_deg - p_square_deg) > e.macro_params.time_step / 10)

In [ ]:
f_square_deg[  2,  28,  10], p_square_deg[  2,  28,  10]

In [ ]:
for i in range(f_lysis_time.shape[0]):
    if np.any(np.abs(f_lysis_time[i] - p_lysis_time[i]) > e.macro_params.time_step):
        print(i)

In [ ]:
f_deg[0, :, 7790]

In [ ]:
f_x_f = np.empty(e.macro_params.total_edges, dtype=float)
f_y_f = np.empty(e.macro_params.total_edges, dtype=float)
p_x_f = np.empty(e.macro_params.total_edges, dtype=float)
p_y_f = np.empty(e.macro_params.total_edges, dtype=float)
for k in range(e.macro_params.total_edges):
    i, j = f_edge_index[k]
    f_x_f[k], f_y_f[k] = plot_coords(i, j)
    i, j = p_edge_index[true_edge][k]
    p_x_f[k], p_y_f[k] = plot_coords(i, j)

In [ ]:
d_x = (rng.random(size=e.macro_params.total_molecules) - 0.5) / 2.5
d_y = (rng.random(size=e.macro_params.total_molecules) - 0.5) / 2.5
f_x_m = np.empty((e.macro_params.total_trials, e.macro_params.number_of_saves, e.macro_params.total_molecules), dtype=float)
f_y_m = np.empty((e.macro_params.total_trials, e.macro_params.number_of_saves, e.macro_params.total_molecules), dtype=float)
p_x_m = np.empty((e.macro_params.total_trials, e.macro_params.number_of_saves, e.macro_params.total_molecules), dtype=float)
p_y_m = np.empty((e.macro_params.total_trials, e.macro_params.number_of_saves, e.macro_params.total_molecules), dtype=float)
for run in range(e.macro_params.total_trials):
    for t in range(e.macro_params.number_of_saves):
        for k in range(e.macro_params.total_molecules):
            i, j = f_edge_index[f_mol_location[run, t, k]]
            f_x_m[run, t, k], f_y_m[run, t, k] = plot_coords(i, j)
            i, j = p_edge_index[p_mol_location[run, t, k]]
            p_x_m[run, t, k], p_y_m[run, t, k] = plot_coords(i, j)
        f_x_m[run, t] += d_x
        f_y_m[run, t] += d_y
        p_x_m[run, t] += d_x
        p_y_m[run, t] += d_y

In [ ]:
ne = f_edge_index[f_mol_location[0]] != p_edge_index[p_mol_location[0]]
np.argwhere(ne)

In [ ]:
f_edge_index[f_mol_location[0,100,21]], p_edge_index[p_mol_location[0,100,21]]

In [ ]:
f_mol_tracker = np.fromfile(os.path.join(e.os_path, "m_tracker" + f_file_code), dtype=np.int32)

In [ ]:
f_mol_tracker = f_mol_tracker.reshape(f_mol_tracker.size // 2, 2)

In [ ]:
p_mol_tracker = np.load(os.path.join(e.os_path, "m_tracker" + p_file_code))

In [ ]:
p_mol_tracker.shape, f_mol_tracker.shape

In [ ]:
np.argwhere(f_edge_index[f_mol_tracker[:38246, 1]] != p_edge_index[p_mol_tracker[:38246,1]])

In [ ]:
f_edge_index[143], 290438*e.macro_params.time_step

In [ ]:
for i in range(27260,38246):
    print(f_mol_tracker[i, 0], 
          f_edge_index[f_mol_tracker[i, 1]],
          p_mol_tracker[i, 0],
          p_edge_index[p_mol_tracker[i,1]])

In [ ]:
np.append(p_tsave, f_tsave, axis=0).T

In [ ]:
p_mol_location

In [ ]:
f_edge_index[7821], p_edge_index[7840]